In [17]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense


In [18]:
nltk.download('punkt_tab')
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [19]:
# For demonstration, use a small sample dataset
# You can replace this with your own CSV file containing text and labels
data = pd.DataFrame({
    'text': [
        "I love this movie! It's amazing and exciting.",
        "This film was terrible, I hated it.",
        "Absolutely fantastic! Best movie ever.",
        "I do not like this movie, it was boring.",
        "An excellent performance by the lead actor."
    ],
    'label': [1, 0, 1, 0, 1]  # 1=positive, 0=negative sentiment
})

X = data['text']
y = data['label']

print("Sample data:")
print(data)


Sample data:
                                            text  label
0  I love this movie! It's amazing and exciting.      1
1            This film was terrible, I hated it.      0
2         Absolutely fantastic! Best movie ever.      1
3       I do not like this movie, it was boring.      0
4    An excellent performance by the lead actor.      1


In [20]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text

X_cleaned = X.apply(clean_text)
print("Cleaned text sample:")
print(X_cleaned)


Cleaned text sample:
0    i love this movie its amazing and exciting
1             this film was terrible i hated it
2          absolutely fantastic best movie ever
3        i do not like this movie it was boring
4    an excellent performance by the lead actor
Name: text, dtype: object


In [21]:
def tokenize_remove_stopwords(text):
    tokens = word_tokenize(text)
    filtered_tokens = [w for w in tokens if w not in stop_words]
    return ' '.join(filtered_tokens)

X_processed = X_cleaned.apply(tokenize_remove_stopwords)
print("Processed text sample:")
print(X_processed)


Processed text sample:
0             love movie amazing exciting
1                     film terrible hated
2    absolutely fantastic best movie ever
3                       like movie boring
4        excellent performance lead actor
Name: text, dtype: object


In [22]:
def tokenize_remove_stopwords(text):
    tokens = word_tokenize(text)
    filtered_tokens = [w for w in tokens if w not in stop_words]
    return ' '.join(filtered_tokens)

X_processed = X_cleaned.apply(tokenize_remove_stopwords)
print("Processed text sample:")
print(X_processed)


Processed text sample:
0             love movie amazing exciting
1                     film terrible hated
2    absolutely fantastic best movie ever
3                       like movie boring
4        excellent performance lead actor
Name: text, dtype: object


In [23]:
tfidf = TfidfVectorizer(max_features=100)
X_tfidf = tfidf.fit_transform(X_processed)

print("TF-IDF shape:", X_tfidf.shape)


TF-IDF shape: (5, 17)


In [24]:
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)


X_train shape: (4, 17)
X_test shape: (1, 17)


In [25]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Embedding, Dropout
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

# For RNN/LSTM, we need sequences instead of TF-IDF
tokenizer = Tokenizer(num_words=1000, oov_token="<OOV>")
tokenizer.fit_on_texts(X_processed)
sequences = tokenizer.texts_to_sequences(X_processed)
X_seq = pad_sequences(sequences, padding='post', maxlen=20)

# Split sequences
X_train_seq, X_test_seq, y_train_seq, y_test_seq = train_test_split(
    X_seq, y, test_size=0.2, random_state=42
)

# Build LSTM model
model = Sequential([
    Embedding(input_dim=1000, output_dim=64, input_length=20),
    LSTM(64),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [26]:
history = model.fit(
    X_train_seq, y_train_seq,
    epochs=10,
    batch_size=2,
    validation_data=(X_test_seq, y_test_seq)
)


Epoch 1/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 4s 407ms/step - accuracy: 0.8333 - loss: 0.6911 - val_accuracy: 0.0000e+00 - val_loss: 0.7306
Epoch 2/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step - accuracy: 0.6667 - loss: 0.6791 - val_accuracy: 0.0000e+00 - val_loss: 0.7568
Epoch 3/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - accuracy: 0.8333 - loss: 0.6543 - val_accuracy: 0.0000e+00 - val_loss: 0.8013
Epoch 4/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step - accuracy: 0.8333 - loss: 0.6306 - val_accuracy: 0.0000e+00 - val_loss: 0.8503
Epoch 5/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 150ms/step - accuracy: 0.6667 - loss: 0.6505 - val_accuracy: 0.0000e+00 - val_loss: 0.8969
Epoch 6/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step - accuracy: 0.6667 - loss: 0.6428 - val_accuracy: 0.0000e+00 - val_loss: 0.9574
Epoch 7/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 141ms/step - accuracy: 0.8333 - loss: 0.5665 - val_accuracy: 0.0000e+00 - val_loss: 1.0535
Epoch 8/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 135ms/step - accuracy: 0.8333 - loss: 0.5381 - val_acc

In [27]:
loss, accuracy = model.evaluate(X_test_seq, y_test_seq)
print("Test Accuracy:", accuracy)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.0000e+00 - loss: 1.3255
Test Accuracy: 0.0
